# Phase A v2 — Definitive Baseline (FULL CICIoT2023)
### *Trustworthiness Under Compression: Calibration and Explanation Faithfulness of Lightweight IoT Intrusion Detectors*

**This is the do-it-once baseline.** Differences from v1:
- **Full dataset** (no sampling) on a **High-RAM runtime** — fixes rare-class starvation (Web/BruteForce precision collapse seen at 2%).
- **Tempered class weights** (√-inverse frequency) — lifts rare classes *without* the carpet-bombing that fully-balanced weights cause, and gentler on calibration (which is half the thesis).
- **Processed-data caching** — CSVs are parsed once into a compressed parquet; re-runs load the cache, so you never reprocess 46M rows again.

Everything else from the plan is preserved: 8 explicit classes incl. benign · locked stratified split · scaler on train only · seeds 0–4 with **seed 0 pre-registered as anchor** · per-class + macro reporting.

> **Runtime:** set **Runtime → Change runtime type → GPU + High-RAM** before running. Full-data 5-seed training is a once-off ~1–2 h job on GPU.

## 0. Setup, reproducibility, paths

In [ ]:
import os, sys, gc, glob, json, random, time, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd, sklearn, tensorflow as tf

for n,m in [('python',sys.version.split()[0]),('numpy',np.__version__),
            ('pandas',pd.__version__),('sklearn',sklearn.__version__),
            ('tensorflow',tf.__version__)]:
    print(f"{n:<12}: {m}")
print("GPU:", tf.config.list_physical_devices('GPU') or "none (use GPU runtime!)")
# High-RAM check
try:
    import psutil; print("RAM (GB):", round(psutil.virtual_memory().total/1e9,1))
except Exception: pass

In [ ]:
def seed_everything(seed:int):
    os.environ["PYTHONHASHSEED"]=str(seed)
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

GLOBAL_DATA_SEED = 42          # locked split only
SEEDS = [0,1,2,3,4]            # M0 ensemble
ANCHOR_SEED = 0                # pre-registered anchor

from google.colab import drive; drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/Trustworthiness_IoT'
PATHS = {k:f'{ROOT}/{k}' for k in ['data','splits','models','results','logs']}
PATHS['raw']=f'{ROOT}/data/raw'; PATHS['cache']=f'{ROOT}/data/cache'
for p in PATHS.values(): os.makedirs(p, exist_ok=True)
print("Dirs ready under", ROOT)

## 1. Labels → 8 classes (verify against your files)

In [ ]:
LABEL_MAP = {
 'DDoS-RSTFINFlood':'DDoS','DDoS-PSHACK_Flood':'DDoS','DDoS-SYN_Flood':'DDoS',
 'DDoS-UDP_Flood':'DDoS','DDoS-TCP_Flood':'DDoS','DDoS-ICMP_Flood':'DDoS',
 'DDoS-SynonymousIP_Flood':'DDoS','DDoS-ACK_Fragmentation':'DDoS',
 'DDoS-UDP_Fragmentation':'DDoS','DDoS-ICMP_Fragmentation':'DDoS',
 'DDoS-HTTP_Flood':'DDoS','DDoS-SlowLoris':'DDoS',
 'DoS-UDP_Flood':'DoS','DoS-TCP_Flood':'DoS','DoS-SYN_Flood':'DoS','DoS-HTTP_Flood':'DoS',
 'Mirai-greeth_flood':'Mirai','Mirai-greip_flood':'Mirai','Mirai-udpplain':'Mirai',
 'Recon-PingSweep':'Recon','Recon-OSScan':'Recon','Recon-PortScan':'Recon',
 'Recon-HostDiscovery':'Recon','VulnerabilityScan':'Recon',
 'DNS_Spoofing':'Spoofing','MITM-ArpSpoofing':'Spoofing',
 'BrowserHijacking':'Web','Backdoor_Malware':'Web','XSS':'Web',
 'Uploading_Attack':'Web','SqlInjection':'Web','CommandInjection':'Web',
 'DictionaryBruteForce':'BruteForce',
 'BenignTraffic':'Benign',
}
CLASSES = ['Benign','DDoS','DoS','Mirai','Recon','Spoofing','Web','BruteForce']
class_to_idx = {c:i for i,c in enumerate(CLASSES)}; BENIGN_IDX = class_to_idx['Benign']
print("8 classes:", CLASSES)

## 2. Full-data load with caching

First run: scans labels → reconciles → parses all CSVs (downcast to float32) → saves one compressed parquet. Later runs: loads the parquet directly (fast).

In [ ]:
CACHE = f"{PATHS['cache']}/ciciot_processed.parquet"
csv_files = sorted(glob.glob(f"{PATHS['raw']}/*.csv"))

if os.path.exists(CACHE):
    print("Loading cached parquet (no reprocessing)…")
    df = pd.read_parquet(CACHE)
else:
    assert csv_files, f"No CSVs in {PATHS['raw']}. Download CICIoT2023 there first."
    # detect label column from first file
    head = pd.read_csv(csv_files[0], nrows=5)
    LABEL_COL = 'label' if 'label' in head.columns else ('Label' if 'Label' in head.columns else head.columns[-1])
    print("Label column:", LABEL_COL, "|", len(csv_files), "CSVs")

    # pass 1: reconcile labels cheaply (label column only)
    seen=set()
    for f in csv_files:
        seen |= set(pd.read_csv(f, usecols=[LABEL_COL])[LABEL_COL].astype(str).str.strip().unique())
    unmapped=[u for u in sorted(seen) if u not in LABEL_MAP]
    assert not unmapped, f"Fix LABEL_MAP for: {unmapped}"
    print(f"All {len(seen)} raw labels mapped. Parsing full data…")

    # pass 2: full parse, downcast, map labels
    parts=[]
    for i,f in enumerate(csv_files):
        d=pd.read_csv(f)
        d['y']=d[LABEL_COL].astype(str).str.strip().map(LABEL_MAP)
        d=d.drop(columns=[LABEL_COL])
        for c in d.columns:
            if c!='y' and d[c].dtype=='float64': d[c]=d[c].astype('float32')
        parts.append(d)
        if (i+1)%25==0: print(f"  {i+1}/{len(csv_files)}")
    df=pd.concat(parts, ignore_index=True); del parts; gc.collect()
    df.to_parquet(CACHE, compression='snappy')
    print("Cached →", CACHE)

print("Full shape:", df.shape)

In [ ]:
dist = df['y'].value_counts().reindex(CLASSES)
share=(dist/dist.sum()*100).round(3)
print("Class distribution (FULL):")
for c in CLASSES: print(f"  {c:<10}{int(dist[c]):>12,}{share[c]:>9}%")
print("\nImbalance ratio (max/min): %.0f:1" % (dist.max()/dist.min()))

## 3. Locked stratified split + train-only scaler

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

X = df.drop(columns=['y']).apply(pd.to_numeric, errors='coerce')
X = X.replace([np.inf,-np.inf], np.nan).fillna(0.0).astype('float32')
y = df['y'].map(class_to_idx).astype('int64').values
del df; gc.collect()
print("X:", X.shape)

idx=np.arange(len(y))
tr,tmp=train_test_split(idx,test_size=0.30,stratify=y,random_state=GLOBAL_DATA_SEED)
va,te =train_test_split(tmp,test_size=0.50,stratify=y[tmp],random_state=GLOBAL_DATA_SEED)
np.savez(f"{PATHS['splits']}/split_primary.npz",train=tr,val=va,test=te)
print(f"Split  train {len(tr):,} | val {len(va):,} | test {len(te):,}")

scaler=StandardScaler().fit(X.iloc[tr])
Xtr=scaler.transform(X.iloc[tr]).astype('float32'); ytr=y[tr]
Xva=scaler.transform(X.iloc[va]).astype('float32'); yva=y[va]
Xte=scaler.transform(X.iloc[te]).astype('float32'); yte=y[te]
joblib.dump(scaler,f"{PATHS['models']}/scaler_primary.joblib")
del X; gc.collect()
N_FEATURES,N_CLASSES=Xtr.shape[1],len(CLASSES)
print("features:",N_FEATURES,"| classes:",N_CLASSES)

## 4. Tempered class weights (√-inverse frequency)

Balanced weights ∝ 1/freq overcorrect on a ~2000:1 imbalance and crush rare-class precision. The √ temper compresses that pressure (a 2000:1 ratio becomes ~45:1), lifting rare classes without carpet-bombing — and distorts probabilities far less, which protects the calibration measurements downstream.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
balanced=compute_class_weight('balanced',classes=np.arange(N_CLASSES),y=ytr)
tempered=np.sqrt(balanced)              # √-inverse-frequency temper
tempered=tempered/ tempered.mean()      # normalise to mean 1
class_weight={i:float(w) for i,w in enumerate(tempered)}
print(f"{'class':<10}{'balanced':>10}{'tempered':>10}")
for i,c in enumerate(CLASSES):
    print(f"{c:<10}{balanced[i]:>10.3f}{tempered[i]:>10.3f}")

## 5. Base architecture + 5-seed M0 (seed 0 = anchor)

In [ ]:
def build_mlp(nf,nc,seed):
    seed_everything(seed); init=tf.keras.initializers.GlorotUniform(seed=seed)
    m=tf.keras.Sequential([
        tf.keras.layers.Input(shape=(nf,)),
        tf.keras.layers.Dense(128,activation='relu',kernel_initializer=init),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3,seed=seed),
        tf.keras.layers.Dense(64,activation='relu',kernel_initializer=init),
        tf.keras.layers.Dropout(0.2,seed=seed),
        tf.keras.layers.Dense(nc,activation='softmax',kernel_initializer=init),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

for s in SEEDS:
    print(f"\n=== M0 seed {s}{'  [ANCHOR]' if s==ANCHOR_SEED else ''} ===")
    seed_everything(s); model=build_mlp(N_FEATURES,N_CLASSES,s)
    es=tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
    rl=tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',patience=3,factor=0.5)
    model.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=60,batch_size=8192,
              class_weight=class_weight,callbacks=[es,rl],verbose=2)
    model.save(f"{PATHS['models']}/M0_seed{s}.keras")
print("\nAll 5 M0 checkpoints saved.")

In [ ]:
# ANCHOR PRE-REGISTRATION (plan §3.1) — written before any trust/explanation metric
rec={'anchor_seed':ANCHOR_SEED,
 'rule':'lowest seed index; fixed by rule prior to analysis; not informed by any downstream metric',
 'all_seeds':SEEDS,
 'roles':{'anchor':'seed 0 — reference for M1-M3 attribution comparison',
          'null_band':'all 5 seeds pairwise — Control 1 retraining variance'},
 'weighting':'tempered sqrt-inverse-frequency, mean-normalised',
 'split_seed':GLOBAL_DATA_SEED,'data':'full CICIoT2023',
 'timestamp_utc':datetime.now(timezone.utc).isoformat()}
json.dump(rec,open(f"{PATHS['logs']}/anchor_preregistration.json",'w'),indent=2)
print(json.dumps(rec,indent=2))

## 6. Per-class sanity (macro-F1, per-class recall, benign FPR)

In [ ]:
from sklearn.metrics import f1_score,recall_score,precision_score,classification_report
anchor=tf.keras.models.load_model(f"{PATHS['models']}/M0_seed{ANCHOR_SEED}.keras")
pred=anchor.predict(Xva,batch_size=8192,verbose=0).argmax(1)
mf1=f1_score(yva,pred,average='macro')
benign_fpr=float(np.mean(pred[yva==BENIGN_IDX]!=BENIGN_IDX))
print(f"Anchor seed {ANCHOR_SEED} on VAL")
print(f"  macro-F1   : {mf1:.4f}")
print(f"  benign FPR : {benign_fpr:.4f}")
print(classification_report(yva,pred,target_names=CLASSES,digits=3))

## Done — definitive baseline

You now have: full-data locked split, train-only scaler, 5 M0 checkpoints, seed-0 anchor pre-registered, and a cached parquet so this never reprocesses. **Compare the Web/BruteForce precision here against the 2% run** — full data + tempered weights should lift them substantially.

**Next: Phase B** — TFLite int8 (M1), pruning ×2 (M2), distilled student (M3) from the seed-0 anchor; Pi benchmarking; adaptive-binned ECE + signed over-confidence gap per class.